In [1]:
import sys
from pathlib import Path
from preprocessing.factory import PipelineFactory
from utils import DataExplorer, build_classification_dataset, inspect_pdf_preprocessing
import spacy
from evaluation import ExtractionEvaluator

/Users/tynka/Documents/pdf-converter/.venv/lib/python3.12/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [3]:


cwd = Path.cwd()
project_dir = cwd if (cwd / "src").is_dir() else (cwd.parent if cwd.name == "src" else cwd)
src_dir = project_dir / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))



data_dir = project_dir / "negatywne"
explorer = DataExplorer(data_dir)

inspect_pdf_preprocessing(
    pdf_path=data_dir / "przyklad_01" / "odwolanie.pdf",
    explorer=explorer,
)


PLIK: odwolanie.pdf


FileNotFoundError: Plik PDF nie istnieje: /Users/tynka/Documents/pdf-converter/negatywne/przyklad_01/odwolanie.pdf

In [4]:


cwd = Path.cwd()
project_dir = cwd if (cwd / "src").is_dir() else (cwd.parent if cwd.name == "src" else cwd)
src_dir = project_dir / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

In [6]:


nlp = spacy.load("pl_core_news_sm", disable=["ner", "parser"])

metadata_path = build_classification_dataset(
    source_dir=data_dir,
    output_file=project_dir / "dataset_neg" / "dataset_neg.jsonl",
    metadata_dir=project_dir / "dataset_neg" / "metadata_neg",
    explorer=explorer,
    pipeline=PipelineFactory.build_for_tfidf(nlp),
    verbose=True,
)

print(f"Gotowe! Metadane zapisane w: {metadata_path}")

[1/45] [POMINIĘTO - brak etykiety]: (generated)Broszura informacyjna MF - Struktura logiczna JPK_FA(4).pdf
[2/45] [POMINIĘTO - brak etykiety]: (generated)Broszura informacyjna MF - Struktura logiczna JPK_MAG(2).pdf
[3/45] [POMINIĘTO - brak etykiety]: (generated)Broszura informacyjna MF - Struktura logiczna JPK_WB(1).pdf
[4/45] [POMINIĘTO - brak etykiety]: (generated)Decyzja Ministra Sportu i Turystyki ws. zatwierdzenia zmian Statutu PZL.pdf
[5/45] [POMINIĘTO - brak etykiety]: (generated)Decyzja Naczelnika Urzędu Skarbowego ws. rozłożenia na raty zaległości podatkowej.pdf
[6/45] [POMINIĘTO - brak etykiety]: (generated)Decyzja Państwowego Powiatowego Inspektora Sanitarnego (Sanepid).pdf
[7/45] [POMINIĘTO - brak etykiety]: (generated)Decyzja Prezesa UOKiK - Praktyki naruszające zbiorowe interesy konsumentów.pdf
[8/45] [POMINIĘTO - brak etykiety]: (generated)Decyzja Prezesa URE - Koncesja na wytwarzanie energii elektrycznej z OZE.pdf
[9/45] [POMINIĘTO - brak etykiety]: (generated)Dec

In [9]:
import logging
logging.getLogger("pdfminer").setLevel(logging.ERROR)

neg_meta_records = []

with out_neg.open("w", encoding="utf-8") as f:
    for i, pdf_path in enumerate(pdfs, 1):
        print(f"[{i}/{len(pdfs)}] START: {pdf_path.name}", flush=True)
        try:
            raw = explorer.read_pdf(pdf_path)
            clean = clean_pipe.run(raw)
            record_id = uuid.uuid4().hex
            record = {
                "id": record_id,
                "text": nlp_pipe.run(raw),
                "text_clean": clean,
                "entities": extract_entities(clean),
                "label": "negatywne",
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()

            neg_meta_records.append({
                "id": record_id,
                "source_pdf": str(pdf_path.resolve()),
                "relative_path": str(pdf_path.relative_to(neg_dir)),
                "label": "negatywne",
            })
        except Exception as e:
            print(f"  [BŁĄD] {pdf_path.name}: {e}", flush=True)

meta_neg_path = project_dir / "dataset_neg" / "metadata_neg.json"
meta_neg_path.write_text(
    json.dumps({
        "dataset_file": str(out_neg.resolve()),
        "labels": ["negatywne"],
        "total_files": len(neg_meta_records),
        "files": neg_meta_records,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Zapisano metadane neg: {meta_neg_path}")

[1/45] START: (generated)Broszura informacyjna MF - Struktura logiczna JPK_FA(4).pdf
[2/45] START: (generated)Broszura informacyjna MF - Struktura logiczna JPK_MAG(2).pdf
[3/45] START: (generated)Broszura informacyjna MF - Struktura logiczna JPK_WB(1).pdf
[4/45] START: (generated)Decyzja Ministra Sportu i Turystyki ws. zatwierdzenia zmian Statutu PZL.pdf
[5/45] START: (generated)Decyzja Naczelnika Urzędu Skarbowego ws. rozłożenia na raty zaległości podatkowej.pdf
[6/45] START: (generated)Decyzja Państwowego Powiatowego Inspektora Sanitarnego (Sanepid).pdf
[7/45] START: (generated)Decyzja Prezesa UOKiK - Praktyki naruszające zbiorowe interesy konsumentów.pdf
[8/45] START: (generated)Decyzja Prezesa URE - Koncesja na wytwarzanie energii elektrycznej z OZE.pdf
[9/45] START: (generated)Decyzja Prezesa UTK - Świadectwo bezpieczeństwa zarządcy infrastruktury kolejowej.pdf
[10/45] START: (generated)Decyzja Urzędu Patentowego RP o udzieleniu patentu na wynalazek.pdf
[11/45] START: (g

In [5]:
main_jsonl = project_dir / "dataset" / "dataset.jsonl"
merged_jsonl = project_dir / "dataset" / "dataset_all.jsonl"

part1 = main_jsonl.read_text(encoding="utf-8").strip()
part2 = out_neg.read_text(encoding="utf-8").strip()
merged_jsonl.write_text(f"{part1}\n{part2}\n", encoding="utf-8")

main_meta = json.loads((project_dir / "dataset" / "metadata" / "metadata.json").read_text(encoding="utf-8"))
all_files = main_meta["files"] + neg_meta_records
all_labels = sorted(set(main_meta["labels"]) | {"negatywne"})

merged_meta_path = project_dir / "dataset" / "metadata" / "metadata_all.json"
merged_meta_path.write_text(
    json.dumps({
        "dataset_file": str(merged_jsonl.resolve()),
        "labels": all_labels,
        "total_files": len(all_files),
        "files": all_files,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Sklejono: {merged_jsonl} oraz {merged_meta_path}")

NameError: name 'out_neg' is not defined

In [5]:
evaluator = ExtractionEvaluator()
report = evaluator.evaluate_training_from_metadata(
    dataset_jsonl=project_dir / "dataset" / "dataset.jsonl",
    metadata_json=project_dir / "dataset" / "metadata" / "metadata.json",
    save_predictions_dir=project_dir / "dataset" / "predictions",
    # only_labels={"przelew"},  # odkomentuj, żeby testować tylko 1 klasę naraz
)
report.print_summary(max_diffs=25)

=== EWALUACJA (182 plików) ===
Overall field accuracy: 46.89% (1356/2892)

[UWAGA] Brak wzorcowego .json dla 58 plików (pierwsze 5):
  - /Users/tynka/Documents/pdf-converter/hacknation-2026-main/dane/przyklad_03/potwierdzenie_przelewu.json
  - /Users/tynka/Documents/pdf-converter/hacknation-2026-main/dane/przyklad_04/potwierdzenie_przelewu.json
  - /Users/tynka/Documents/pdf-converter/hacknation-2026-main/dane/przyklad_05/potwierdzenie_przelewu.json
  - /Users/tynka/Documents/pdf-converter/hacknation-2026-main/dane/przyklad_06/potwierdzenie_przelewu.json
  - /Users/tynka/Documents/pdf-converter/hacknation-2026-main/dane/przyklad_07/potwierdzenie_przelewu.json

--- Per klasa dokumentu ---
  odwolanie         : 47.95% (80 plików)
  pelnomocnictwo    : 45.56% (80 plików)
  przelew           : 62.12% (22 plików)

--- Najsłabsze pola ---
  odwolanie:OdwolanieKIO.czynnosc_dzien        : 0.00% (0/80)
  odwolanie:OdwolanieKIO.czynnosc_opis         : 0.00% (0/80)
  odwolanie:OdwolanieKIO.czynno